# Steam Game Recommender System with Spark MLlib

## 1. Introduction
This notebook builds a collaborative filtering recommender system for Steam, the online video game store, using Apache Spark MLlib on Databricks. The goal is to explore the data, train a model that predicts which games a user is likely to enjoy based on their play history, and evaluate and tune that model.

The dataset, steam-200k.csv, contains 200,000 rows recording the purchase and play behaviour of Steam members. Each row is a single interaction between a user and a game: the user ID, the game name, the behaviour (purchase or play) and a value, which is 1 for a purchase or the number of hours played for a play.

The data is loaded into a Spark DataFrame, given meaningful column names and explored with both PySpark DataFrames and Spark SQL. Because ALS needs integer IDs for both users and items, integer game IDs are generated inside Spark with MLlib's StringIndexer. Play hours are log-transformed to compress their heavily skewed distribution and used as implicit feedback. A baseline ALS model is trained and evaluated, a grid search over 15 hyperparameter combinations is run and tracked with MLflow, and personalised recommendations are generated for an individual user and for all users in batch.

## 2. Data Import and Preprocessing
Before any modelling could begin, the data needed to be located, inspected, loaded into Spark, and given meaningful column names so that it could be queried and transformed reliably. This section describes each of those preparatory steps.


### 2.1 Locating the Files

Before reading the data, the files are checked to confirm they exist and are accessible, using two different mechanisms. The %fs magic command lists the file metadata directly in the notebook output, confirming that steam-200k.csv and the supplementary games.csv are present. The Python call dbutils.fs.ls() returns the same information as a list of FileInfo objects that can be used programmatically.

In [0]:
dbutils.fs.ls('/Volumes/teaching/datasets/assignment_2/task_2/')

[FileInfo(path='dbfs:/Volumes/teaching/datasets/assignment_2/task_2/games.csv', name='games.csv', size=143639, modificationTime=1771957901000),
 FileInfo(path='dbfs:/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv', name='steam-200k.csv', size=8059447, modificationTime=1771957902000)]

In [0]:
%fs
ls '/Volumes/teaching/datasets/assignment_2/task_2/'

path,name,size,modificationTime
dbfs:/Volumes/teaching/datasets/assignment_2/task_2/games.csv,games.csv,143639,1771957901000
dbfs:/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv,steam-200k.csv,8059447,1771957902000


### 2.2 Importing Libraries and Loading the Dataset

With the files verified, the necessary PySpark libraries were imported and the CSV was read into a Spark DataFrame:

The pyspark.sql.functions module, imported under the alias F, provides built-in Spark functions.
The header=False parameter is specified because the CSV file does not contain column names in its first row; the four columns are unnamed. Setting inferSchema=True instructs Spark to automatically determine the data type of each column by sampling the file, so that numeric columns such as the user ID and play hours are read as numeric types rather than strings. After loading, count() confirmed that 200,000 records were loaded successfully, and printSchema() confirmed the inferred column types.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Path to the dataset in Unity Catalog
data_path = "/Volumes/teaching/datasets/assignment_2/task_2/steam-200k.csv"

# Defining a static data frame on the files 
steamDF = spark.read.csv(
    data_path,
    header=False,
    inferSchema=True
)

print(f"Total rows in the dataset: {steamDF.count():,}")
steamDF.printSchema()
display(steamDF)

Total rows in the dataset: 200,000
root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: double (nullable = true)



_c0,_c1,_c2,_c3
151603712,The Elder Scrolls V Skyrim,purchase,1.0
151603712,The Elder Scrolls V Skyrim,play,273.0
151603712,Fallout 4,purchase,1.0
151603712,Fallout 4,play,87.0
151603712,Spore,purchase,1.0
151603712,Spore,play,14.9
151603712,Fallout New Vegas,purchase,1.0
151603712,Fallout New Vegas,play,12.1
151603712,Left 4 Dead 2,purchase,1.0
151603712,Left 4 Dead 2,play,8.9


### 2.3 Viewing the Structure of the Data

The toDF() method renames all four columns in a single operation:

The new names follow the dataset description. userId holds the unique member identifier, gameName holds the game title, behaviour indicates whether the row represents a purchase or play event, and value holds either 1 for purchase rows or the number of hours played for play rows. Renaming columns at this early stage means all subsequent code and SQL queries use self-documenting names. A display() call following the rename confirmed the column names were applied correctly.

In [0]:
# Renaming the columns

steamDF = steamDF.toDF("userId", "gameName", "behaviour", "value")
display(steamDF)

userId,gameName,behaviour,value
151603712,The Elder Scrolls V Skyrim,purchase,1.0
151603712,The Elder Scrolls V Skyrim,play,273.0
151603712,Fallout 4,purchase,1.0
151603712,Fallout 4,play,87.0
151603712,Spore,purchase,1.0
151603712,Spore,play,14.9
151603712,Fallout New Vegas,purchase,1.0
151603712,Fallout New Vegas,play,12.1
151603712,Left 4 Dead 2,purchase,1.0
151603712,Left 4 Dead 2,play,8.9


## 3.0 Exploratory Data Analysis Using PySpark DataFrames

Before building a model it is essential to understand the structure, scale, and statistical properties of the data. 

### 3.1 Behaviour Type Counts

The first EDA step counts the number of rows belonging to each behaviour type using the groupBy().count() pattern. 

In [0]:
# 1. Behaviour type counts 

steamDF.groupBy("behaviour") \
       .count() \
       .orderBy("behaviour") \
       .display(truncate=False)

behaviour,count
play,70489
purchase,129511


Databricks visualization. Run in Databricks to view.

### 3.2 Unique Users and Games

The select().distinct().count() chain is used to count unique values in the userId and gameName columns without materialising the full distinct sets. 

In [0]:
#  2. Unique users and unique game names 

n_users = steamDF.select("userId").distinct().count()
n_games = steamDF.select("gameName").distinct().count()
print(f"Unique users : {n_users:,}")
print(f"Unique games : {n_games:,}")

Unique users : 12,393
Unique games : 5,155


### 3.3 Play-Hours Summary Statistics

The describe() method computes the count, mean, standard deviation, minimum and maximum of the value column, restricted to play rows only. The mean is about 49 hours, but the standard deviation is about 229 hours and the maximum is 11,754 hours, which shows the distribution is heavily right-skewed: most users play a game for a few hours, while a small number play for thousands.

In [0]:
# 3. Play-hours summary statistics 

steamDF.filter(F.col("behaviour") == "play") \
       .select("value") \
       .describe() \
       .display(truncate=False)

summary,value
count,70489
mean,48.87806324391008
stddev,229.335235996813
min,0.1
max,11754.0


### 3.4 Top 10 Games by Total Play Hours

This step aggregates total play hours per game using groupBy().agg() with both F.sum() and F.count() in a single pass, which is more efficient than performing two separate groupBy operations. 


In [0]:
# --- 4. Top 10 games by total play hours ---

steamDF.filter(F.col("behaviour") == "play") \
       .groupBy("gameName") \
       .agg(
           F.round(F.sum("value"), 0).alias("total_hours"),
           F.count("*").alias("num_players")
       ) \
       .orderBy(F.col("total_hours").desc()) \
       .limit(10) \
       .display(truncate=False)

gameName,total_hours,num_players
Dota 2,981685.0,4841
Counter-Strike Global Offensive,322772.0,1377
Team Fortress 2,173673.0,2323
Counter-Strike,134261.0,568
Sid Meier's Civilization V,99821.0,554
Counter-Strike Source,96076.0,715
The Elder Scrolls V Skyrim,70889.0,677
Garry's Mod,49725.0,666
Call of Duty Modern Warfare 2 - Multiplayer,42010.0,291
Left 4 Dead 2,33597.0,801


### 3.5 Games Purchased Per User

This step applies describe() to the count of games purchased per user after a groupBy("userId").count(). 


In [0]:
#  5. Games owned per user 

steamDF.filter(F.col("behaviour") == "purchase") \
       .groupBy("userId") \
       .count() \
       .select(F.col("count").alias("games_purchased")) \
       .describe() \
       .show()

+-------+-----------------+
|summary|  games_purchased|
+-------+-----------------+
|  count|            12393|
|   mean|10.45033486645687|
| stddev|35.86426882024551|
|    min|                1|
|    max|             1075|
+-------+-----------------+



## 4.0 Verifying the Exploratory Analysis with Spark SQL
The same summaries are re-run with Spark SQL against a temporary view of the data. Spark SQL and DataFrame operations run on the same execution engine, so matching results confirm that the Python code is correct and that the dataset is understood properly.

In [0]:
# Register a temporary view so the same data is queryable via SQL
steamDF.createOrReplaceTempView("steam_data")

In [0]:
%sql
-- Verify 1: behaviour type counts
-- Expected: play = 70,489  |  purchase = 129,511
SELECT behaviour,
       COUNT(*) AS record_count
FROM   steam_data
GROUP  BY behaviour
ORDER  BY behaviour;

behaviour,record_count
play,70489
purchase,129511


Databricks visualization. Run in Databricks to view.

In [0]:
%sql
-- Verify 2: unique user and game counts
-- Expected: users = 12,393  |  games = 5,155
SELECT COUNT(DISTINCT userId)   AS unique_users,
       COUNT(DISTINCT gameName) AS unique_games
FROM   steam_data;

unique_users,unique_games
12393,5155


In [0]:
%sql
-- Verify 3: top 10 games by total play hours
-- Expected: Dota 2 first with ~982,000 hours
SELECT   gameName,
         ROUND(SUM(value), 0) AS total_hours,
         COUNT(*)             AS num_players
FROM     steam_data
WHERE    behaviour = 'play'
GROUP BY gameName
ORDER BY total_hours DESC
LIMIT    10;

gameName,total_hours,num_players
Dota 2,981685.0,4841
Counter-Strike Global Offensive,322772.0,1377
Team Fortress 2,173673.0,2323
Counter-Strike,134261.0,568
Sid Meier's Civilization V,99821.0,554
Counter-Strike Source,96076.0,715
The Elder Scrolls V Skyrim,70889.0,677
Garry's Mod,49725.0,666
Call of Duty Modern Warfare 2 - Multiplayer,42010.0,291
Left 4 Dead 2,33597.0,801


In [0]:
%sql
-- Verify 4: play-hours statistics
-- Expected: mean ~48.9, max 11,754
SELECT ROUND(MIN(value),   2) AS min_hours,
       ROUND(AVG(value),   2) AS avg_hours,
       ROUND(MAX(value),   2) AS max_hours,
       ROUND(STDDEV(value), 2) AS stddev_hours,
       COUNT(*)                 AS total_play_rows
FROM   steam_data
WHERE  behaviour = 'play';

min_hours,avg_hours,max_hours,stddev_hours,total_play_rows
0.1,48.88,11754.0,229.34,70489


In [0]:
%sql
-- Verify 5: distribution of games purchased per user
-- Expected: mean ~10.45, max 1,075
SELECT ROUND(MIN(games_purchased),    0) AS min_owned,
       ROUND(AVG(games_purchased),    2) AS avg_owned,
       ROUND(MAX(games_purchased),    0) AS max_owned,
       COUNT(*)                            AS total_users
FROM (
    SELECT   userId,
             COUNT(*) AS games_purchased
    FROM     steam_data
    WHERE    behaviour = 'purchase'
    GROUP BY userId
) AS user_counts;

min_owned,avg_owned,max_owned,total_users
1,10.45,1075,12393


## 5.0 MLflow Setup and Autologging

MLflow is imported and autologging is enabled for PySpark ML. Autologging records the parameters, metrics and artefacts of every training run automatically, which makes it easy to compare experiments without manual logging code. Non-error MLflow messages are suppressed for cleaner output, and a named experiment is created so that every run from this notebook is grouped together in the Databricks Experiments UI.

In [0]:
import mlflow
import logging

# Suppress non-error MLflow messages for cleaner notebook output
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Enable automatic logging of every PySpark ML fit() call
mlflow.pyspark.ml.autolog()

# Build a unique experiment name from the current notebook path
ctx       = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb        = ctx.notebookPath().get().split("/")[2]
safe_path = f"dbfs:/FileStore/{nb}/"

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri("databricks-uc")

experiment_name = f"/Shared/{nb}_steam_recommender"

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name)

mlflow.set_experiment(experiment_name)
print(f"MLflow experiment ready: {experiment_name}")

## 6. Generating Integer Game IDs Using StringIndexer

ALS in Spark MLlib needs integer IDs for both users and items. The userId column is already numeric, but games are identified only by name. StringIndexer, a standard MLlib transformer, maps each unique game name to an integer index. With stringOrderType='frequencyDesc', index 0 goes to the most frequent game, index 1 to the second most frequent, and so on. With handleInvalid='keep', any game name not seen before is given a special index instead of causing an error. This generates the IDs entirely within Spark, without relying on an external lookup file.

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.types import IntegerType

# Fit StringIndexer on the FULL dataset - every game name gets a mapping
game_indexer = StringIndexer(
    inputCol        = "gameName",
    outputCol       = "gameId",
    stringOrderType = "frequencyDesc",  # most-played game → index 0
    handleInvalid   = "keep"
)

indexer_model = game_indexer.fit(steamDF)
steamDF        = indexer_model.transform(steamDF)

# Cast gameId from DoubleType (StringIndexer default) to IntegerType (ALS requirement)
steamDF = steamDF.withColumn("gameId", F.col("gameId").cast(IntegerType()))

print("Schema after StringIndexer transformation:")
steamDF.printSchema()



Schema after StringIndexer transformation:
root
 |-- userId: integer (nullable = true)
 |-- gameName: string (nullable = true)
 |-- behaviour: string (nullable = true)
 |-- value: double (nullable = true)
 |-- gameId: integer (nullable = true)



### 6.1 Cross-Checking Against games.csv

As an independent check, the supplementary games.csv file is loaded and its row count (5,155) is compared with the number of distinct game IDs generated by StringIndexer (also 5,155), confirming that every game received a unique integer ID. The file is used only for this check, not to create the IDs.

In [0]:
games_path = "/Volumes/teaching/datasets/assignment_2/task_2/games.csv"

gamesDF = spark.read.csv(
    games_path,
    header=False,
    inferSchema=True
)

print(f"games.csv - rows: {gamesDF.count():,}   columns: {len(gamesDF.columns)}")
gamesDF.printSchema()
display(gamesDF.limit(10))

games.csv - rows: 5,155   columns: 2
root
 |-- _c0: integer (nullable = true)
 |-- _c1: string (nullable = true)



_c0,_c1
1,The Elder Scrolls V Skyrim
2,Fallout 4
3,Spore
4,Fallout New Vegas
5,Left 4 Dead 2
6,HuniePop
7,Path of Exile
8,Poly Bridge
9,Left 4 Dead
10,Team Fortress 2


In [0]:
# Show the top 10 most-common games with their assigned gameIds

steamDF.select("gameName", "gameId") \
       .distinct() \
       .orderBy("gameId") \
       .display(10, truncate=False)

n_ids = steamDF.select("gameId").distinct().count()
print(f"Unique gameIds generated : {n_ids:,}")
print(f"Rows in games.csv        : {gamesDF.count():,}")
print("→ Both sources cover the same 5,155 unique games")

gameName,gameId
Dota 2,0
Team Fortress 2,1
Counter-Strike Global Offensive,2
Unturned,3
Left 4 Dead 2,4
Counter-Strike Source,5
Counter-Strike,6
Garry's Mod,7
The Elder Scrolls V Skyrim,8
Warframe,9


Unique gameIds generated : 5,155
Rows in games.csv        : 5,155
→ Both sources cover the same 5,155 unique games


### 6.2 Verifying the Game IDs with Spark SQL

In [0]:
# Re-register the updated temp view (now includes gameId column)
steamDF.createOrReplaceTempView("steam_data")

In [0]:
%sql
-- Verify: total distinct gameIds matches games.csv (expected: 5,155)
SELECT COUNT(DISTINCT gameId)   AS unique_game_ids,
       COUNT(DISTINCT gameName) AS unique_game_names
FROM   steam_data;

unique_game_ids,unique_game_names
5155,5155


In [0]:
%sql
-- Verify: lowest gameIds belong to the most-played games
-- (frequencyDesc: gameId 0 should be the most common game in the dataset)
SELECT   gameId,
         gameName,
         COUNT(*) AS appearances
FROM     steam_data
GROUP BY gameId, gameName
ORDER BY appearances DESC
LIMIT    5;

gameId,gameName,appearances
0,Dota 2,9682
1,Team Fortress 2,4646
2,Counter-Strike Global Offensive,2789
3,Unturned,2632
4,Left 4 Dead 2,1752


Databricks visualization. Run in Databricks to view.

## 7. Preparing the ALS Input: Filtering and Log-Transforming Play Hours

There are two behaviour types. Play hours are used as the feedback signal rather than the purchase flag, because hours carry much richer preference information than a value that is always 1. Users have not rated games explicitly, so this is implicit feedback, and the ALS model is configured with implicitPrefs=True, which treats the value as a confidence weight rather than a rating.

The exploratory analysis showed that play hours are heavily right-skewed, with a mean of about 49 hours and a maximum of 11,754 hours. A log1p transform, log(1 + hours), is applied to compress the extreme values and stabilise the confidence signal.

The check below finds 12 duplicate (userId, gameId) pairs out of 70,477, where a user has two play records for the same game. At 0.02% of the data this has a negligible effect on training.

In [0]:
# Filter to play rows only
playDF = steamDF.filter(F.col("behaviour") == "play")

# Apply log1p transform to compress the heavy-tailed hours distribution
playDF = playDF.withColumn("hours_log", F.log1p(F.col("value")))

# Drop rows with nulls in any of the ALS input columns
playDF = playDF.dropna(subset=["userId", "gameId", "hours_log"])

print(f"Play rows for modelling     : {playDF.count():,}")
total    = playDF.count()
distinct = playDF.select("userId", "gameId").distinct().count()
print(f"Distinct (userId,gameId)    : {distinct:,}")
print(f"Duplicate pairs             : {total - distinct:,}")

display(playDF.select(
    "userId", "gameName", "gameId",
    F.round("value", 1).alias("hours_raw"),
    F.round("hours_log", 4).alias("hours_log")
).limit(8))

Play rows for modelling     : 70,489
Distinct (userId,gameId)    : 70,477
Duplicate pairs             : 12


userId,gameName,gameId,hours_raw,hours_log
151603712,The Elder Scrolls V Skyrim,8,273.0,5.6131
151603712,Fallout 4,100,87.0,4.4773
151603712,Spore,332,14.9,2.7663
151603712,Fallout New Vegas,29,12.1,2.5726
151603712,Left 4 Dead 2,4,8.9,2.2925
151603712,HuniePop,867,8.5,2.2513
151603712,Path of Exile,39,8.1,2.2083
151603712,Poly Bridge,1347,7.5,2.1401


In [0]:
# Register play subset as a temp view for SQL verification
playDF.createOrReplaceTempView("play_data")


### 7.1 Verifying the Play Data with Spark SQL

SQL was used to verify the row count after filtering, the absence of duplicate (userId, gameId) pairs, and a sample of the log-transform values to confirm the formula was applied correctly.

In [0]:
%sql
-- Verify row counts and duplicate check on play_data
-- Expected: 70,489 rows, 12 duplicate pairs
SELECT COUNT(*)                            AS total_rows,
       COUNT(DISTINCT userId, gameId)     AS distinct_pairs,
       COUNT(*) - COUNT(DISTINCT userId, gameId) AS duplicates
FROM   play_data;

total_rows,distinct_pairs,duplicates
70489,70477,12


In [0]:
%sql
-- Spot-check log1p transform: verify hours_log = ln(1 + hours_raw)
-- ln(1 + 273) = ln(274) ≈ 5.6131  - expected for Skyrim row
SELECT gameName,
       ROUND(value,     1) AS hours_raw,
       ROUND(hours_log, 4) AS hours_log,
       ROUND(LN(1 + value), 4) AS log1p_check
FROM   play_data
WHERE  userId = 151603712
ORDER  BY hours_raw DESC
LIMIT  5;

gameName,hours_raw,hours_log,log1p_check
The Elder Scrolls V Skyrim,273.0,5.6131,5.6131
Fallout 4,87.0,4.4773,4.4773
Spore,14.9,2.7663,2.7663
Fallout New Vegas,12.1,2.5726,2.5726
Left 4 Dead 2,8.9,2.2925,2.2925


## 8. Splitting Data into Training and Test Sets

The play data is split into a training set (80%) and a held-out test set (20%) with randomSplit() and a fixed seed for reproducibility. The test set is kept completely separate and is only used for final evaluation. It is never seen during hyperparameter tuning, which uses a further 75/25 split inside the training data.

In [0]:
# 80% training / 20% test - seed=42 ensures reproducible splits
(trainingDF, testDF) = playDF.randomSplit([0.8, 0.2], seed=42)

print(f"Training rows : {trainingDF.count():,}")
print(f"Test rows     : {testDF.count():,}")
print(f"Split ratio   : {trainingDF.count()/playDF.count()*100:.1f}% / {testDF.count()/playDF.count()*100:.1f}%")

Training rows : 56,518
Test rows     : 13,971
Split ratio   : 80.2% / 19.8%


In [0]:
# Register both splits as temp views for SQL inspection
trainingDF.createOrReplaceTempView("training_data")
testDF.createOrReplaceTempView("test_data")

In [0]:
%sql
-- Verify split sizes and that there is no data leakage between splits
-- (same userId can appear in both - that is expected and correct for ALS)
SELECT 'training'             AS split,
       COUNT(*)               AS rows,
       COUNT(DISTINCT userId) AS unique_users,
       COUNT(DISTINCT gameId) AS unique_games
FROM   training_data
UNION ALL
SELECT 'test'                 AS split,
       COUNT(*)               AS rows,
       COUNT(DISTINCT userId) AS unique_users,
       COUNT(DISTINCT gameId) AS unique_games
FROM   test_data;

split,rows,unique_users,unique_games
training,56518,9988,3379
test,13971,4505,2058


## 9. Training a Baseline ALS Model

implicitPrefs=True switches ALS from the explicit rating formulation to the implicit feedback formulation, which is correct for play-hour data. coldStartStrategy='drop' removes predictions for users or games that were not seen during training, rather than returning NaN values that would break the evaluation. The baseline hyperparameters (rank=10, maxIter=10, regParam=0.1) are reasonable starting values that the grid search varies later. Because MLflow autologging is enabled, this run is recorded automatically.

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol           = "userId",
    itemCol           = "gameId",
    ratingCol         = "hours_log",   # implicit confidence signal
    implicitPrefs     = True,           # play hours = implicit feedback
    coldStartStrategy = "drop",         # drop NaN for unseen users/games
    rank              = 10,
    maxIter           = 10,
    regParam          = 0.1,
    seed              = 42
)

# Train baseline - MLflow autolog records parameters and metrics
baseline_model = als.fit(trainingDF)
print("Baseline ALS model trained successfully.")

Baseline ALS model trained successfully.


## 10. Evaluating the Baseline Model

The baseline model is evaluated with Root Mean Square Error (RMSE) on the held-out test set. The predictions are also registered as a temporary view so the RMSE can be recomputed independently with Spark SQL, confirming that the evaluator and the manual formula agree.

The test set has 13,971 rows, of which 12,312 receive a prediction. The other 1,659 rows belong to users or games that do not appear in the training data (the cold-start problem) and are dropped.

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# Generate predictions on the test set; drop NaN rows (cold-start)
baseline_preds = baseline_model.transform(testDF).dropna()

evaluator = RegressionEvaluator(
    metricName    = "rmse",
    labelCol      = "hours_log",
    predictionCol = "prediction"
)

baseline_rmse = evaluator.evaluate(baseline_preds)
print(f"Baseline RMSE (PySpark evaluator): {baseline_rmse:.5f}")

# Register predictions as a temp view for SQL verification
baseline_preds.createOrReplaceTempView("baseline_predictions")

# Preview sample predictions
baseline_preds.select(
    "userId", "gameName",
    F.round("hours_log",   4).alias("actual"),
    F.round("prediction", 4).alias("predicted")
).orderBy("userId").display(8, truncate=False)

Baseline RMSE (PySpark evaluator): 2.40550


userId,gameName,actual,predicted
5250,Deus Ex Human Revolution,4.1431,0.0193
76767,Age of Empires II HD Edition,2.6462,0.0586
76767,Banished,3.2189,-0.0186
76767,Call of Duty Modern Warfare 3,2.8273,0.6617
76767,Half-Life,0.7885,0.1462
76767,Thief Deadly Shadows,1.4816,0.0017
86540,Age of Empires II HD Edition,0.5306,0.1114
86540,Killer is Dead,1.8405,0.0127
86540,Left 4 Dead 2,2.8565,0.3946
86540,XCOM Enemy Unknown,3.6376,0.1083


In [0]:
%sql
-- Verify RMSE manually using the SQL RMSE formula:
-- RMSE = SQRT( AVG( (predicted - actual)^2 ) )
-- Expected: 2.40550 (matching the PySpark evaluator above)
SELECT ROUND(SQRT(AVG((prediction - hours_log) * (prediction - hours_log))), 5)
       AS rmse_sql,
       COUNT(*) AS prediction_rows
FROM   baseline_predictions;

rmse_sql,prediction_rows
2.4055,12312


## 11. Hyperparameter Tuning with Grid Search

The grid search explores 15 combinations (3 rank values x 5 regParam values). rank sets the number of latent factors: a higher rank can capture more complex user-game relationships but increases the risk of overfitting. The values tested (5, 10, 15) range from a simple to a moderately complex model. regParam sets the strength of L2 regularisation: higher values guard against overfitting, lower values let the model fit the training data more closely. The values tested (0.001 to 0.1) span two orders of magnitude.

TrainValidationSplit is used instead of CrossValidator because it makes a single 75/25 train-validation split inside the training data rather than k-fold cross-validation, which is far cheaper on large data while still giving a reliable estimate of validation performance. With MLflow autologging enabled, all 15 runs are recorded with their hyperparameters and validation RMSE.

In [0]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# Build the hyperparameter grid: 3 ranks × 5 regParams = 15 combinations
paramGrid = ParamGridBuilder() \
    .addGrid(als.rank,     [5, 10, 15]) \
    .addGrid(als.regParam, [0.001, 0.005, 0.01, 0.05, 0.1]) \
    .build()

print(f"Hyperparameter combinations: {len(paramGrid)}")

# TrainValidationSplit: 75% train / 25% validation within trainingDF
tvs = TrainValidationSplit() \
    .setSeed(42) \
    .setTrainRatio(0.75) \
    .setEstimatorParamMaps(paramGrid) \
    .setEstimator(als) \
    .setEvaluator(evaluator)

print("Running grid search - please wait...")
gridsearch_model = tvs.fit(trainingDF)
print("Grid search complete.")

Hyperparameter combinations: 15
Running grid search - please wait...
Grid search complete.


## 12. Selecting the Best Model and Final Evaluation

bestModel holds the ALS model with the lowest validation RMSE. It is then evaluated on the held-out test set, which was never used for training or validation.

The best combination was rank 10 and regParam 0.01, with a test RMSE of 2.40399 against the baseline's 2.40550, an improvement of only 0.0015. Tuning rank and regParam made almost no difference, and the reason is the metric rather than the model. With implicitPrefs=True, ALS predicts a preference score that sits roughly between 0 and 1, while the test labels are log play hours that range up to about 9. RMSE therefore mostly measures the gap in scale between the two, which no choice of rank or regParam can close. RMSE is a reasonable sanity check here, but it does not measure what matters for a recommender: whether the games a user actually plays rank near the top of their recommendations. The conclusion sets out how to evaluate that properly.

In [0]:
# Extract best model and print its hyperparameters
bestModel = gridsearch_model.bestModel

print("=== Best Model Hyperparameters ===")
print(f"  Rank     : {bestModel.rank}")
print(f"  RegParam : {bestModel._java_obj.parent().getRegParam()}")
print(f"  MaxIter  : {bestModel._java_obj.parent().getMaxIter()}")

# Evaluate on the held-out test set
best_preds = bestModel.transform(testDF).dropna()
best_rmse  = evaluator.evaluate(best_preds)

print(f"\n  Baseline RMSE  : {baseline_rmse:.5f}")
print(f"  Best Model RMSE: {best_rmse:.5f}")
print(f"  Improvement    : {baseline_rmse - best_rmse:.5f}")

# Register for SQL verification
best_preds.createOrReplaceTempView("best_predictions")

=== Best Model Hyperparameters ===
  Rank     : 10
  RegParam : 0.01
  MaxIter  : 10

  Baseline RMSE  : 2.40550
  Best Model RMSE: 2.40399
  Improvement    : 0.00150


## 13. Generating Recommendations

### 13.1 Recommendations for a Specific User

Personalised recommendations are generated for user 151603712 in three steps. First, the games this user has already played are retrieved. Second, a DataFrame of every game they have not played is built with a left anti-join, which keeps only rows from the left DataFrame that have no match in the right one. A literal userId column is added with F.lit() so the DataFrame matches the schema the model expects. Third, the best model scores every unplayed game, and the results are sorted by predicted score to give the top recommendations. The SQL cell that follows reproduces the same top 10 as a cross-check.

In [0]:
example_uid = 151603712

# Games already played by this user
played = playDF.filter(F.col("userId") == example_uid) \
               .select("gameId", "gameName",
                       F.round("value", 1).alias("hours_played"))

print(f"Games already played by user {example_uid}:")
played.orderBy(F.col("hours_played").desc()).show(truncate=False)

# All games NOT yet played by this user
game_lookup = steamDF.select("gameId", "gameName").distinct()
played_ids  = played.select("gameId")
unplayed    = game_lookup.join(played_ids, on="gameId", how="left_anti") \
                         .withColumn("userId", F.lit(example_uid))

# Generate predictions for unplayed games
user_preds = bestModel.transform(unplayed).dropna()
user_preds.createOrReplaceTempView("user_predictions")

print(f"\nTop 10 recommendations for user {example_uid} (PySpark):")
user_preds.orderBy(F.col("prediction").desc()) \
          .select("gameName", F.round("prediction", 4).alias("score")) \
          .show(10, truncate=False)

Games already played by user 151603712:
+------+-------------------------------------+------------+
|gameId|gameName                             |hours_played|
+------+-------------------------------------+------------+
|8     |The Elder Scrolls V Skyrim           |273.0       |
|100   |Fallout 4                            |87.0        |
|332   |Spore                                |14.9        |
|29    |Fallout New Vegas                    |12.1        |
|4     |Left 4 Dead 2                        |8.9         |
|867   |HuniePop                             |8.5         |
|39    |Path of Exile                        |8.1         |
|1347  |Poly Bridge                          |7.5         |
|49    |Left 4 Dead                          |3.3         |
|1     |Team Fortress 2                      |2.8         |
|55    |Tomb Raider                          |2.5         |
|604   |The Banner Saga                      |2.0         |
|51    |Dead Island Epidemic                 |1.4         |


In [0]:
%sql
-- Verify top 10 recommendations using SQL
-- Expected: same ranking as PySpark output above
SELECT   gameName,
         ROUND(prediction, 4) AS score
FROM     user_predictions
ORDER BY prediction DESC
LIMIT    10;

gameName,score
Terraria,0.6082
Borderlands 2,0.5256
Portal 2,0.5125
Sid Meier's Civilization V,0.4575
Warframe,0.3706
PAYDAY 2,0.318
Portal,0.3112
Don't Starve,0.2846
Far Cry 3,0.2647
Torchlight II,0.2565


### 13.2 Recommendations for All Users

recommendForAllUsers(10) generates the top 10 recommendations for every user in a single distributed pass. Each row of the output holds a userId and an array of 10 (gameId, rating) pairs, which explode() turns into one row per recommendation. Game names are joined back from the StringIndexer mapping so the results are readable. The SQL cell that follows retrieves the recommendations for one user to confirm the batch and individual approaches agree.

In [0]:
from pyspark.sql.functions import explode

# Generate top-10 for all users simultaneously
all_user_recs = bestModel.recommendForAllUsers(10)

# Flatten the nested recommendations array
flat_recs = all_user_recs \
    .withColumn("rec", explode("recommendations")) \
    .select(
        "userId",
        F.col("rec.gameId").alias("gameId"),
        F.round(F.col("rec.rating"), 4).alias("predicted_score")
    )

# Join game names back in from the StringIndexer mapping
game_lookup = steamDF.select("gameId", "gameName").distinct()
flat_recs   = flat_recs.join(game_lookup, on="gameId", how="left")

# Register as temp view
flat_recs.createOrReplaceTempView("all_recommendations")

print(f"Total recommendation rows: {flat_recs.count():,}")
print(f"(= {flat_recs.select('userId').distinct().count():,} users × 10 recs each)")

flat_recs.filter(F.col("userId") == 5250) \
         .orderBy(F.col("predicted_score").desc()) \
         .show(10, truncate=False)

Total recommendation rows: 99,880
(= 9,988 users × 10 recs each)
+------+------+---------------+--------------------------+
|gameId|userId|predicted_score|gameName                  |
+------+------+---------------+--------------------------+
|1     |5250  |0.9625         |Team Fortress 2           |
|0     |5250  |0.8122         |Dota 2                    |
|5     |5250  |0.4679         |Counter-Strike Source     |
|11    |5250  |0.4639         |Sid Meier's Civilization V|
|14    |5250  |0.3521         |Portal                    |
|15    |5250  |0.3417         |Portal 2                  |
|16    |5250  |0.3284         |Half-Life 2               |
|7     |5250  |0.3057         |Garry's Mod               |
|9     |5250  |0.2894         |Warframe                  |
|22    |5250  |0.263          |Borderlands 2             |
+------+------+---------------+--------------------------+



In [0]:
%sql
-- Verify total recommendation count and retrieve top-10 for user 5250
SELECT   userId,
         gameName,
         predicted_score
FROM     all_recommendations
WHERE    userId = 5250
ORDER BY predicted_score DESC
LIMIT    10;

userId,gameName,predicted_score
5250,Team Fortress 2,0.9625
5250,Dota 2,0.8122
5250,Counter-Strike Source,0.4679
5250,Sid Meier's Civilization V,0.4639
5250,Portal,0.3521
5250,Portal 2,0.3417
5250,Half-Life 2,0.3284
5250,Garry's Mod,0.3057
5250,Warframe,0.2894
5250,Borderlands 2,0.263


## 14. Conclusion

This notebook used Apache Spark MLlib on Databricks to build a collaborative filtering recommender on 200,000 Steam user interactions. The data was loaded and explored, with every summary cross-checked in Spark SQL. Integer game IDs were generated inside Spark with StringIndexer, play hours were log-transformed to tame their skew, and an ALS model was trained on implicit feedback. A grid search over 15 hyperparameter combinations was tracked with MLflow, and recommendations were produced for an individual user and for all 9,988 users in the training data.

The recommendations are sensible. A user whose library is dominated by Skyrim, Fallout and other story-driven games is recommended Terraria, Borderlands 2, Portal 2 and Civilization V. Another user is recommended popular multiplayer titles such as Team Fortress 2, Dota 2 and Counter-Strike Source, which shows the model also leans on overall popularity.

### Limitations and next steps

- **Use ranking metrics.** Implicit ALS outputs preference scores, not hours, so RMSE against log hours mostly measures a difference in scale, which is why tuning barely moved it. Evaluating with ranking metrics such as precision at k, mean average precision at k and NDCG at k (available in MLlib's RankingEvaluator) would show how often the games a user actually plays appear in their top recommendations.
- **Tune alpha.** In implicit ALS, alpha controls how strongly more play time increases confidence. It was left at its default here and is likely to matter more than regParam.
- **Combine signals.** Purchases without any recorded play could be added as a weaker signal, rather than discarded.
- **Handle cold start.** Users and games with very few interactions could be filtered out of training, or given popularity-based recommendations instead.
- **Remove the 12 duplicate user-game pairs**, for example by summing their hours, before training.